In [2]:
import pandas as pd
import numpy as np

In [3]:
account = pd.read_csv('../ravenstack_accounts.csv')
churn = pd.read_csv('../ravenstack_churn_events.csv')
subscription = pd.read_csv('../ravenstack_subscriptions.csv')

In [4]:
churn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   churn_event_id            600 non-null    object 
 1   account_id                600 non-null    object 
 2   churn_date                600 non-null    object 
 3   reason_code               600 non-null    object 
 4   refund_amount_usd         600 non-null    float64
 5   preceding_upgrade_flag    600 non-null    bool   
 6   preceding_downgrade_flag  600 non-null    bool   
 7   is_reactivation           600 non-null    bool   
 8   feedback_text             452 non-null    object 
dtypes: bool(3), float64(1), object(5)
memory usage: 30.0+ KB


In [17]:
churn.head(5)


,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
0,C-816288,A-c37cab,2024-10-27,pricing,4.03,False,False,False,switched to competitor
1,C-5a81e7,A-37f969,2024-06-25,support,96.45,True,False,False,NaN
2,C-a174be,A-b07346,2024-11-12,budget,0.00,False,False,False,missing features
3,C-accb39,A-1e50e0,2023-11-01,budget,54.94,False,False,False,switched to competitor
4,C-92f889,A-956988,2024-12-30,unknown,0.00,False,True,True,too expensive


In [5]:
subscription.head(5)
# subscription.info()

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,NaN,Pro,17,833,9996,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,NaN,Enterprise,62,0,0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,2024-01-10,NaN,Enterprise,27,5373,64476,False,False,False,False,monthly,True


In [6]:
account.head(5)
account.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   account_id       500 non-null    object
 1   account_name     500 non-null    object
 2   industry         500 non-null    object
 3   country          500 non-null    object
 4   signup_date      500 non-null    object
 5   referral_source  500 non-null    object
 6   plan_tier        500 non-null    object
 7   seats            500 non-null    int64 
 8   is_trial         500 non-null    bool  
 9   churn_flag       500 non-null    bool  
dtypes: bool(2), int64(1), object(7)
memory usage: 32.4+ KB


In [7]:
# checking if one customer have multiple account
subscription.groupby('account_id')['subscription_id'].count()

account_id
A-00bed1    10
A-00cac8     9
A-0158bb     6
A-016043    11
A-019782     9
            ..
A-fe79a5    10
A-ff3c73     9
A-ff79f2     9
A-ffc04f    15
A-ffdfd5    10
Name: subscription_id, Length: 500, dtype: int64

In [8]:
# How many unique customers
subscription['account_id'].nunique()

500

# Data cleaning and analysis

In [19]:
# for accounts data set
account['signup_date'] = pd.to_datetime(account['signup_date'])

In [8]:
churn['churn_date'] = pd.to_datetime(churn['churn_date'])

# total churn events
total_churn = (churn['account_id']).nunique()
print(f"Total churn events: {total_churn}")

# churn reason
churn.groupby('reason_code')['account_id'].count()

# adding month column
churn['month'] = churn['churn_date'].dt.month_name()

# which month have high churn
churn.groupby('month')['reason_code'].count().sort_values(ascending = False)

# cutomer who never churn
all_account = set(account['account_id'])
churn_cust = set(churn['account_id'])

never_churn_cust = all_account - churn_cust
print(f"customer who never churn : {len(never_churn_cust)}")

Total churn events: 352
customer who never churn : 148


In [14]:
subscription['start_date'] = pd.to_datetime(subscription['start_date'])
subscription['end_date'] = pd.to_datetime(subscription['end_date'])

earliest_date = subscription['start_date'].min()
latest_date = subscription['start_date'].max()


print(f"Earliest Date : {earliest_date}")
print(f"latest Date : {latest_date}")

min_monthly_rev = subscription['mrr_amount'].min()
max_monthly_rev = subscription['mrr_amount'].max()

min_annually_rev = subscription['arr_amount'].min()
max_annually_rev = subscription['arr_amount'].max()

print(f"minimum monthly revenue : {min_monthly_rev}")
print(f"maximum monthly revenue : {max_monthly_rev}")
print(f"minimun annual revenue : {min_annually_rev}")
print(f"maximun annual revenue : {max_annually_rev}")

# add status column
subscription['status'] = np.where(subscription['end_date'].isna(),"Active","Inactive") 

active_subscription = (subscription['status'] == 'Active').sum()
print(f"Total active subscription : {active_subscription}")

inactive_subscription = (subscription['status'] == 'Inactive').sum()
print(f"Total inactive subscription : {inactive_subscription}")

# checking customer on each plan
subscription.groupby('plan_tier')['account_id'].nunique()

subscription.groupby('status')['account_id'].nunique()


Earliest Date : 2023-01-09 00:00:00
latest Date : 2024-12-31 00:00:00
minimum monthly revenue : 0
maximum monthly revenue : 33830
minimun annual revenue : 0
maximun annual revenue : 405960
Total active subscription : 4514
Total inactive subscription : 486


status
Active      500
Inactive    312
Name: account_id, dtype: int64